<a href="https://colab.research.google.com/github/Shirish-12105/AI-HIRING-PREDICTION-SYSTEM/blob/main/ai_hirinig_prediction_system%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# AI-Based Hiring Prediction System
# End-to-End Machine Learning Project
# ============================================================

# ✅ TASK 1: Load and Understand Dataset
import pandas as pd
import numpy as np

df = pd.read_csv('/content/AI-Based Hiring Prediction System.csv')

print("Head:\n", df.head())
print("Tail:\n", df.tail())
print("Sample:\n", df.sample(5))

# ✅ TASK 2: Data Inspection
print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nTarget Distribution:\n", df['Recruiter Decision'].value_counts())
print("\nStatistics:\n", df.describe())

# ✅ TASK 3: Data Cleaning
df = df.drop(columns=['Resume_ID', 'Name', 'AI Score (0-100)'])

df['Recruiter Decision'] = df['Recruiter Decision'].map({'Hire': 1, 'Reject': 0})

df['Skills']         = df['Skills'].fillna('')
df['Certifications'] = df['Certifications'].fillna('')
df['Job Role']       = df['Job Role'].fillna('')

# ✅ TASK 4: Text Feature Engineering
df['combined_text'] = (
    df['Skills'] + " " +
    df['Certifications'] + " " +
    df['Job Role']
).str.lower()

df['combined_text'] = df['combined_text'].str.replace(r'[^a-z\s]', '', regex=True)

# ✅ TASK 5: Convert Text to Numerical (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)
X_text = tfidf.fit_transform(df['combined_text']).toarray()

# ✅ TASK 6: Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Education_Encoded'] = le.fit_transform(df['Education'])

# ✅ TASK 7: Feature & Target Separation
# Numeric features: Experience, Salary, Projects (to be scaled)
# Categorical: Education_Encoded (NOT scaled)
X_num = df[['Experience (Years)', 'Salary Expectation ($)', 'Projects Count']].values
X_edu = df[['Education_Encoded']].values
y     = df['Recruiter Decision'].values

# ✅ TASK 8: Train-Test Split
from sklearn.model_selection import train_test_split

X_combined = np.hstack((X_num, X_edu, X_text))

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

# ✅ TASK 9: Feature Scaling (only first 3 numeric columns)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[:, :3] = scaler.fit_transform(X_train[:, :3])
X_test[:, :3]  = scaler.transform(X_test[:, :3])

# ✅ TASK 10: Model Training
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM":                 SVC(probability=True, random_state=42),
    "KNN":                 KNeighborsClassifier()
}

# ✅ TASK 11: Evaluation & Comparison
from sklearn.metrics import accuracy_score, classification_report

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)

    print(f"\n{'='*40}")
    print(f"{name} Accuracy: {acc:.2f}")
    print(classification_report(y_test, preds))

    results.append([name, round(acc, 2)])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"])
print("\nModel Comparison:\n", results_df)

# ✅ TASK 13: Prediction Function
best_model = models["Random Forest"]

def predict_hiring(skills, exp, edu, cert, role, projects, salary):
    """
    Predict hiring decision for a candidate.

    Parameters:
        skills   (str)   : Candidate skills e.g. "Python SQL"
        exp      (int)   : Years of experience
        edu      (str)   : Education level e.g. "B.Tech"
        cert     (str)   : Certifications e.g. "AWS"
        role     (str)   : Job role e.g. "Data Scientist"
        projects (int)   : Number of projects
        salary   (int)   : Salary expectation in USD

    Returns:
        decision   (str)   : "Hire" or "Reject"
        confidence (float) : Model confidence score
    """
    # Text processing
    combined  = (str(skills) + " " + str(cert) + " " + str(role)).lower()
    combined  = __import__('re').sub(r'[^a-z\s]', '', combined)
    text_vec  = tfidf.transform([combined]).toarray()

    # Education encoding
    try:
        edu_enc = le.transform([edu])[0]
    except ValueError:
        print(f"Warning: Education '{edu}' not seen during training. Defaulting to 0.")
        edu_enc = 0

    # Numeric scaling
    num        = np.array([[exp, salary, projects]], dtype=float)
    num_scaled = scaler.transform(num)

    # Combine all features: [scaled_num | edu_encoded | tfidf]
    final_input = np.hstack((
        num_scaled,
        np.array([[edu_enc]]),
        text_vec
    ))

    pred       = best_model.predict(final_input)[0]
    prob       = best_model.predict_proba(final_input)[0]
    confidence = round(float(np.max(prob)), 2)

    return ("Hire" if pred == 1 else "Reject"), confidence


# ✅ Test the prediction function
decision, confidence = predict_hiring(
    skills   = "Python SQL TensorFlow",
    exp      = 5,
    edu      = "B.Tech",
    cert     = "AWS",
    role     = "Data Scientist",
    projects = 4,
    salary   = 80000
)

print(f"\n{'='*40}")
print(f"Prediction : {decision}")
print(f"Confidence : {confidence * 100:.1f}%")

# ✅ TASK 14: Conclusion
print("""
============================================================
CONCLUSION
============================================================
This project successfully developed an AI-based hiring
prediction system using machine learning techniques.
The dataset contained both structured and unstructured data,
requiring preprocessing steps such as handling missing values,
encoding categorical variables, and converting text into
numerical features using TF-IDF.

Multiple models were trained, including Logistic Regression,
Random Forest, SVM, and KNN. Among these, Random Forest
performed the best due to its ability to handle mixed data
types and capture complex patterns.

This project provided hands-on experience in data
preprocessing, feature engineering, model training, and
evaluation. It also demonstrated how AI can be applied in
real-world HR systems to automate resume screening and
improve hiring efficiency.
============================================================
""")


Head:
    Resume_ID              Name                                        Skills  \
0          1        Ashley Ali                      TensorFlow, NLP, Pytorch   
1          2      Wesley Roman  Deep Learning, Machine Learning, Python, SQL   
2          3     Corey Sanchez         Ethical Hacking, Cybersecurity, Linux   
3          4  Elizabeth Carney                   Python, Pytorch, TensorFlow   
4          5        Julie Hill                              SQL, React, Java   

   Experience (Years) Education                Certifications  \
0                  10      B.Sc                           NaN   
1                  10       MBA                     Google ML   
2                   1       MBA  Deep Learning Specialization   
3                   7    B.Tech                 AWS Certified   
4                   4       PhD                           NaN   

                Job Role Recruiter Decision  Salary Expectation ($)  \
0          AI Researcher               Hire       